# Assignment 1 — EDA ve Baseline Modeller
**XACLE Challenge 2027** (IEEE ICASSP 2027 – The 2nd X-to-Audio Alignment Challenge)

Görev: Bir ses kaydı ile metin açıklaması arasındaki anlamsal uyum için insanların verdiği puanı tahmin etmek.

İçerik:
1. Kurulum
2. Veri kümesine genel bakış
3. Puan dağılımı
4. Dinleyiciler arası tutarlılık
5. Ses üretim modellerine göre analiz
6. Metin analizi
7. Ses analizi
8. Baseline modeller

## 1. Kurulum

In [ ]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()
REPO_URL = "https://github.com/deafiduck/.-XACLE-IEEE-ICASSP-2027---The-2nd-X-to-Audio-Alignment-Challenge-.git"

if IN_COLAB:
    # Kernel Colab'da çalışıyor: src/ kodunu GitHub'dan çek (önce push etmiş olun!)
    REPO_DIR = Path("/content/repo")
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    REPO_DIR = Path.cwd()

sys.path.insert(0, str(REPO_DIR))
print("IN_COLAB:", IN_COLAB, "| REPO_DIR:", REPO_DIR)

In [ ]:
# Veri kümesinin yeri
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/XACLE")
    LOCAL_DIR = Path("/content/data")
    DATASET = "XACLE_Challenge_2027_development_dataset"
    zip_path = DRIVE_DIR / f"{DATASET}.zip"

    if (LOCAL_DIR / DATASET).exists():
        DATA_ROOT = LOCAL_DIR / DATASET            # bu oturumda zaten açılmış
    elif zip_path.exists():
        # Drive'dan binlerce küçük dosyayı tek tek okumak çok yavaş:
        # zip'i Colab'ın kendi diskine kopyalayıp orada açıyoruz (~1-2 dk)
        LOCAL_DIR.mkdir(parents=True, exist_ok=True)
        subprocess.run(["cp", str(zip_path), str(LOCAL_DIR)], check=True)
        subprocess.run(["unzip", "-q", "-o", str(LOCAL_DIR / zip_path.name),
                        "-x", "__MACOSX/*", "-d", str(LOCAL_DIR)], check=True)
        (LOCAL_DIR / zip_path.name).unlink()
        DATA_ROOT = LOCAL_DIR / DATASET
    else:
        DATA_ROOT = DRIVE_DIR / DATASET            # açılmış klasör Drive'da
else:
    DATA_ROOT = REPO_DIR / "datasets" / "XACLE_Challenge_2027_development_dataset"

print(DATA_ROOT, "| bulundu:", DATA_ROOT.exists())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

from src.data import (load_metadata, load_all, audio_path, audio_info,
                      FILE_COL, TEXT_COL, MODEL_COL, LISTENER_COL, SCORE_COL, SPLITS)
from src.utils import compute_metrics
from src.models import MeanBaseline

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120)

## 2. Veri kümesine genel bakış
Her örnek (instance) bir **(ses, metin)** çiftidir. Öznitelikler: ses dosyası (sayısal sinyal), metin açıklaması (string), üretici model (kategorik). Hedef değişken: ortalama uyum puanı (sayısal).

In [ ]:
df = load_all(DATA_ROOT, average=True)          # ses başına ortalama puan
df_l = load_all(DATA_ROOT, average=False)       # dinleyici bazlı puanlar

print(df.groupby("split").size())
df.head()

In [ ]:
df.info()
print("\nEksik değerler:\n", df.isna().sum())
print("\nTekrar eden satır:", df.duplicated().sum())
print("Tekrar eden dosya adı:", df[FILE_COL].duplicated().sum())
print("Tekrar eden metin:", df[TEXT_COL].duplicated().sum())